# 14 實戰案例 — 練習

用松柏護理之家退伍軍人症資料，獨立完成一份迷你疫調報告。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150


## 題目 1：疫情摘要表

1. 讀取 `data/synthetic/legionella_outbreak.csv`
2. 計算以下指標：
   - 總住民數、感染人數、死亡人數
   - 侵襲率、致死率
   - 住院率（住院人數 / 感染人數）
   - ICU 比例（ICU 人數 / 住院人數）
3. 用 `pd.DataFrame` 做成表格輸出

In [ ]:
# TODO: 讀取資料
# TODO: 計算各項指標
# TODO: 組合成摘要表

## 題目 2：危險因子快速篩查

對以下 4 個暴露因子，各做一個 2×2 表並計算 RR：
- `shower_use`
- `hydrotherapy_use`
- `comorbidity_copd`
- `immunosuppressed`

用 `for` 迴圈一次算完，輸出比較表。
哪個因子的 RR 最大？

In [ ]:
# TODO: 對 4 個因子用 for 迴圈計算 RR
# TODO: 輸出比較表
# TODO: 找出 RR 最大的因子

## 題目 3（挑戰題）：迷你 SitRep

產出一份包含以下 3 張圖的迷你 SitRep：

1. **流行曲線**：每日新增個案數（bar chart），標示高峰日
2. **年齡分布**：感染 vs 未感染的年齡直方圖
3. **樓層侵襲率**：6 個區域（1F-A ~ 3F-B）的侵襲率長條圖

用 `fig, axes = plt.subplots(1, 3)` 排成一列。
在最後印出「行動建議」——根據你的分析，建議優先處理哪個區域？

In [ ]:
# TODO: 畫出 3 張圖
# TODO: 印出行動建議

## 題目 4：諾羅病毒宴會群聚迷你疫調（諾羅病毒情境）

一場宴會後爆發腸胃炎群聚。從賓客的食物暴露與發病 line list 做完整迷你疫調。

1. 畫流行曲線（發病時間），判斷傳播型態
2. 對每種食物計算侵襲率與風險比 RR
3. 找出 RR 最高的可疑食物並做卡方檢定
4. 寫一段結論：可疑感染源與流行曲線的意義

In [ ]:
# 諾羅病毒宴會群聚：150 位賓客的食物暴露與發病 line list
from epi_learning.metrics import attack_rate, risk_ratio
rng = np.random.default_rng(1404)
n = 150
foods = ["生蠔", "沙拉", "甜點", "湯品"]
ate = {f: rng.binomial(1, 0.5, n) for f in foods}
p_ill = (0.05 + 0.7 * ate["生蠔"]).clip(0, 1)     # 生蠔受汙染
ill = rng.binomial(1, p_ill)
onset_hr = np.where(ill == 1, rng.normal(32, 8, n).clip(6, 72), np.nan)  # 諾羅潛伏 ~24-48h
guests = pd.DataFrame({"guest_id": range(1, n + 1), "ill": ill,
                       **{f: ate[f] for f in foods}, "onset_hr": np.round(onset_hr, 0)})
print(f"宴會 {n} 人，發病 {ill.sum()} 人（{ill.mean():.1%}）")

# TODO: 畫流行曲線——把發病者的 onset_hr 以 6 小時為間距做直方圖，判斷是否為 point source
# TODO: 對每種食物計算「有吃 vs 沒吃」的侵襲率與風險比 RR（用 risk_ratio）
# TODO: 找出 RR 最高的可疑食物，並用 scipy.stats 卡方檢定其顯著性
# TODO: 寫一段結論：可疑感染源是什麼？流行曲線形狀支持什麼傳播型態？

## 題目 5：COVID-19 職場群聚調查（COVID-19 情境）

某公司出現 COVID-19 群聚，懷疑一場全員大會是暴露事件。

1. 畫流行曲線（依發病日）
2. 算各部門侵襲率，找出最高風險部門
3. 計算「參加大會」的風險比 RR
4. 寫結論：大會是否為可疑暴露？

In [ ]:
# COVID-19 職場群聚：某公司 200 名員工，一場全員大會為可疑暴露
from epi_learning.metrics import risk_ratio
rng = np.random.default_rng(1405)
n = 200
dept = rng.choice(["業務", "研發", "行政", "客服"], n, p=[0.3, 0.3, 0.2, 0.2])
meeting = rng.binomial(1, np.where(dept == "業務", 0.9, 0.4))   # 業務多半有參加
p_inf = (0.03 + 0.35 * meeting).clip(0, 1)
infected = rng.binomial(1, p_inf)
onset_day = np.where(infected == 1, rng.integers(2, 10, n), -1)   # 會後第幾天發病
staff = pd.DataFrame({"emp_id": range(1, n + 1), "dept": dept,
                      "meeting": meeting, "infected": infected, "onset_day": onset_day})
print(f"公司 {n} 人，確診 {infected.sum()} 人；參加大會者 {meeting.sum()} 人")

# TODO: 畫流行曲線（依 onset_day 統計每日新增）
# TODO: 用 groupby 算各部門的確診數與侵襲率，找出最高風險部門
# TODO: 計算「有參加大會 vs 沒參加」的風險比 RR
# TODO: 寫結論：大會是否為可疑暴露事件？為什麼業務部風險最高？

## 題目 6（挑戰題）：登革熱社區疫情 SitRep（登革熱情境）

某社區登革熱流行，你要產出一份情勢報告（SitRep）。

1. 畫全社區每週流行曲線
2. 算各行政區累計每十萬人發生率，排出熱區
3. 判斷疫情趨勢（上升／持平／下降）
4. 寫一份 3–5 句 SitRep：規模、熱區、趨勢、防治建議

In [ ]:
# 登革熱社區疫情：5 個行政區、10 週的每週病例與人口（挑戰題：寫一份 SitRep）
rng = np.random.default_rng(1406)
districts = ["安南區", "三民區", "北屯區", "板橋區", "中西區"]
pop = {"安南區": 190000, "三民區": 340000, "北屯區": 280000, "板橋區": 550000, "中西區": 78000}
weekly_rate = {"安南區": 3.0, "三民區": 1.2, "北屯區": 0.8, "板橋區": 0.6, "中西區": 1.4}  # /10萬/週
_rows = []
for wk in range(1, 11):
    growth = 1.0 + 0.15 * wk    # 疫情逐週上升
    for d in districts:
        cases = rng.poisson(weekly_rate[d] * growth * pop[d] / 100000)
        _rows.append({"epi_week": wk, "district": d, "cases": cases})
dengue = pd.DataFrame(_rows)
region_pop = pd.DataFrame({"district": districts, "population": [pop[d] for d in districts]})
print(f"登革熱：{dengue['cases'].sum()} 例，10 週 × {len(districts)} 區")

# TODO: 畫全社區的每週流行曲線（groupby epi_week 加總）
# TODO: 合併 region_pop，算各區「累計每十萬人發生率」，排出熱區
# TODO: 用每週病例數判斷疫情趨勢（上升／持平／下降）
# TODO: 寫一份 3–5 句的 SitRep：目前規模、熱區、趨勢、建議的防治行動